In [1]:
"""
Normalizacion radiometrica para mosaico SAR (adaptacion del metodo RPRN,
Liu et al. 2022, IEEE GRSL). SOLO normaliza; NO hace el mosaico (eso va en SNAP).

Modelo: h_i * sigma_i = h_j * sigma_j sobre "candidatos de ajuste" (solape geografico,
sin reflexion especular ni doble rebote), resuelto de forma global con minimos
cuadrados ridge. h_i = ganancia lineal por imagen (equivale al ajuste de Qv del paper;
Sentinel-1 no tiene Qv). No se modela el termino de angulo de incidencia.

Entrada: .dim de SNAP (lee <nombre>.data/<banda>.img) o .tif. La banda debe estar en
escala LINEAL (Sigma0), no en dB. Salida: <nombre>_RN.tif (float32, nodata=0).

Requiere: numpy, scipy, rasterio
"""

from __future__ import annotations

import itertools
import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.windows import Window, from_bounds
from scipy.ndimage import uniform_filter


@dataclass
class Config:
    band_name: str = "Sigma0"   # banda a usar cuando la entrada es .dim
    band: int = 1                  # indice de banda cuando la entrada es .tif
    window_size: int = 9           # ventana de estadisticas locales (impar)
    low_mean_pct: float = 15.0     # percentil bajo de media -> descarta especular
    high_std_pct: float = 85.0     # percentil alto de std -> descarta doble rebote
    min_candidates: int = 500      # minimo de pixeles candidatos por par
    ridge_lambda: float = 1e-3     # regularizacion del sistema
    nodata_value: float = 0.0      # nodata de entrada (SNAP usa 0 por defecto)
    chunk_rows: int = 1024         # filas por bloque al escribir


# --------------------------------------------------------------------------
# Entrada
# --------------------------------------------------------------------------

def resolve_path(path: str, cfg: Config) -> str:
    """.dim -> <nombre>.data/<band_name>.img ; cualquier otro -> tal cual."""
    p = Path(path)
    if p.suffix.lower() != ".dim":
        return str(p)
    img = p.with_suffix(".data") / f"{cfg.band_name}.img"
    if not img.exists():
        disponibles = [q.stem for q in p.with_suffix(".data").glob("*.img")]
        raise FileNotFoundError(f"No existe {img}. Bandas disponibles: {disponibles}")
    return str(img)


def check_image(path: str, cfg: Config):
    with rasterio.open(path) as src:
        if src.crs is None:
            raise ValueError(f"{path}: sin georreferenciacion (CRS). Exportala a GeoTIFF desde SNAP.")
        h, w = min(src.height, 1000), min(src.width, 1000)
        a = src.read(cfg.band, out_shape=(h, w))
    a = a[np.isfinite(a) & (a != cfg.nodata_value)]
    if a.size and (a < 0).mean() > 0.05:
        raise ValueError(f"{path}: parece estar en dB (valores negativos). Usa la banda lineal (Sigma0), no Sigma0_db.")


# --------------------------------------------------------------------------
# Estadisticas locales y candidatos de ajuste
# --------------------------------------------------------------------------

def local_mean_std(arr, window, valid):
    """Media/std locales excluyendo invalidos. 'full' = ventana completamente valida."""
    a = np.where(valid, arr, 0.0)
    cnt = uniform_filter(valid.astype(np.float64), size=window, mode="nearest")
    full = cnt > 0.999
    cnt = np.clip(cnt, 1e-6, None)
    mean = uniform_filter(a, size=window, mode="nearest") / cnt
    mean_sq = uniform_filter(a ** 2, size=window, mode="nearest") / cnt
    std = np.sqrt(np.clip(mean_sq - mean ** 2, 0, None))
    return mean, std, full


def overlap_arrays(path_i, path_j, cfg: Config):
    """Recorta el solape sobre la grilla de i y reproyecta j encima (pixel a pixel)."""
    with rasterio.open(path_i) as si, rasterio.open(path_j) as sj:
        bi = si.bounds
        bj = transform_bounds(sj.crs, si.crs, *sj.bounds)  # mismo CRS antes de intersectar
        left, right = max(bi.left, bj[0]), min(bi.right, bj[2])
        bottom, top = max(bi.bottom, bj[1]), min(bi.top, bj[3])
        if left >= right or bottom >= top:
            return None

        w = from_bounds(left, bottom, right, top, transform=si.transform)
        c0, r0 = max(int(np.floor(w.col_off)), 0), max(int(np.floor(w.row_off)), 0)
        c1 = min(int(np.ceil(w.col_off + w.width)), si.width)
        r1 = min(int(np.ceil(w.row_off + w.height)), si.height)
        if c1 - c0 < 2 or r1 - r0 < 2:
            return None
        win = Window(c0, r0, c1 - c0, r1 - r0)

        a = si.read(cfg.band, window=win).astype(np.float64)
        b = np.full(a.shape, np.nan, dtype=np.float64)
        reproject(
            source=rasterio.band(sj, cfg.band), destination=b,
            src_transform=sj.transform, src_crs=sj.crs,
            dst_transform=si.window_transform(win), dst_crs=si.crs,
            src_nodata=cfg.nodata_value, dst_nodata=np.nan,   # nodata no se mezcla en el bilinear
            resampling=Resampling.bilinear,
        )
        return a, b


@dataclass
class PairStats:
    i: int
    j: int
    n: int
    mean_i: float
    mean_j: float
    std_i: float
    std_j: float
    sxx_i: float   # suma x_i^2
    sxx_j: float   # suma x_j^2
    sxy: float     # suma x_i*x_j

    @property
    def log_ratio(self) -> float:
        return float(np.log(self.mean_j / self.mean_i))


def pair_stats(path_i, path_j, i, j, cfg: Config):
    res = overlap_arrays(path_i, path_j, cfg)
    if res is None:
        return None
    a, b = res
    nd = cfg.nodata_value
    valid = np.isfinite(a) & np.isfinite(b) & (a != nd) & (b != nd) & (a > 0) & (b > 0)
    if valid.sum() < cfg.min_candidates:
        return None

    ma, sa, fa = local_mean_std(a, cfg.window_size, valid)
    mb, sb, fb = local_mean_std(b, cfg.window_size, valid)
    core = valid & fa & fb                       # lejos de bordes/nodata
    if core.sum() < cfg.min_candidates:
        return None

    lo_a, lo_b = np.percentile(ma[core], cfg.low_mean_pct), np.percentile(mb[core], cfg.low_mean_pct)
    hi_a, hi_b = np.percentile(sa[core], cfg.high_std_pct), np.percentile(sb[core], cfg.high_std_pct)
    stable = core & (ma > lo_a) & (mb > lo_b) & (sa < hi_a) & (sb < hi_b)  # sin especular ni doble rebote
    if stable.sum() < cfg.min_candidates:
        return None

    x, y = a[stable], b[stable]
    if x.mean() <= 0 or y.mean() <= 0:
        return None
    return PairStats(i, j, int(x.size), float(x.mean()), float(y.mean()), float(x.std()), float(y.std()),
                     float((x ** 2).sum()), float((y ** 2).sum()), float((x * y).sum()))


def build_pairs(paths, cfg: Config):
    pairs = []
    for (i, pi), (j, pj) in itertools.combinations(enumerate(paths), 2):
        p = pair_stats(pi, pj, i, j, cfg)
        if p is not None:
            pairs.append(p)
    return pairs


# --------------------------------------------------------------------------
# Ajuste global (minimos cuadrados ridge, ponderado)
# --------------------------------------------------------------------------

def solve_gains(pairs, n_images, cfg: Config):
    """log h_i - log h_j = log(media_j / media_i), todas las parejas a la vez.
    Peso sqrt(n): mas solape/candidatos = mas peso (el paper pesa por numero de TPs)."""
    A = np.zeros((len(pairs), n_images))
    L = np.zeros(len(pairs))
    for k, p in enumerate(pairs):
        A[k, p.i], A[k, p.j], L[k] = 1.0, -1.0, p.log_ratio
    w = np.sqrt([p.n for p in pairs])
    w = w / w.mean()
    AtA = A.T @ (A * w[:, None]) + cfg.ridge_lambda * np.eye(n_images)
    log_h = np.linalg.solve(AtA, A.T @ (w * L))
    log_h -= log_h.mean()          # ancla: ganancia media geometrica = 1
    return np.exp(log_h)


def compute_metrics(pairs, g, label):
    """Indicadores de la Tabla II del paper (D_mu, D_delta, RMSE por pixel) + D_mu en dB."""
    mi = np.array([p.mean_i * g[p.i] for p in pairs]); mj = np.array([p.mean_j * g[p.j] for p in pairs])
    si = np.array([p.std_i * g[p.i] for p in pairs]); sj = np.array([p.std_j * g[p.j] for p in pairs])
    sse = sum(g[p.i] ** 2 * p.sxx_i - 2 * g[p.i] * g[p.j] * p.sxy + g[p.j] ** 2 * p.sxx_j for p in pairs)
    return {
        "label": label,
        "D_mu": float(np.mean(np.abs(mi - mj))),
        "D_mu_dB": float(np.mean(np.abs(10 * np.log10(mi / mj)))),
        "D_delta": float(np.mean(np.abs(si - sj))),
        "RMSE": float(np.sqrt(sse / sum(p.n for p in pairs))),
    }


# --------------------------------------------------------------------------
# Salida
# --------------------------------------------------------------------------

def apply_gain_and_save(path_in, gain, path_out, cfg: Config):
    with rasterio.open(path_in) as src:
        prof = dict(driver="GTiff", height=src.height, width=src.width, count=1, dtype="float32",
                    crs=src.crs, transform=src.transform, nodata=0.0, compress="deflate",
                    tiled=True, blockxsize=256, blockysize=256, BIGTIFF="IF_SAFER")
        with rasterio.open(path_out, "w", **prof) as dst:
            for r0 in range(0, src.height, cfg.chunk_rows):
                win = Window(0, r0, src.width, min(cfg.chunk_rows, src.height - r0))
                a = src.read(cfg.band, window=win).astype(np.float64)
                ok = np.isfinite(a) & (a != cfg.nodata_value)
                dst.write(np.where(ok, a * gain, 0.0).astype(np.float32), 1, window=win)


# --------------------------------------------------------------------------
# Orquestacion
# --------------------------------------------------------------------------

def run(inputs, output_dir, cfg: Config | None = None):
    cfg = cfg or Config()
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    names = [Path(p).stem for p in inputs]
    paths = [resolve_path(p, cfg) for p in inputs]
    for p in paths:
        check_image(p, cfg)

    print(f"[1/4] Pares de solape y candidatos ({len(paths)} imagenes)...")
    pairs = build_pairs(paths, cfg)
    if not pairs:
        raise RuntimeError("Ningun par con solape valido y suficientes candidatos. "
                           "Revisa CRS/bounds o baja 'min_candidates'.")
    print(f"      {len(pairs)} pares utilizables de {len(paths) * (len(paths) - 1) // 2} posibles.")
    used = {p.i for p in pairs} | {p.j for p in pairs}
    isolated = [names[i] for i in range(len(paths)) if i not in used]
    if isolated:
        print("      AVISO: sin ningun par valido (no se corrigen de forma confiable):", *isolated, sep="\n        - ")

    print("[2/4] Ajuste global de ganancias...")
    g = solve_gains(pairs, len(paths), cfg)
    for n, gi in zip(names, g):
        print(f"      {n}: h = {gi:.4f} ({10 * np.log10(gi):+.2f} dB)")

    print("[3/4] Metricas antes/despues...")
    m0 = compute_metrics(pairs, np.ones_like(g), "antes")
    m1 = compute_metrics(pairs, g, "despues")
    for m in (m0, m1):
        print(f"      {m['label']:8s} D_mu={m['D_mu']:.5f} ({m['D_mu_dB']:.2f} dB)  D_delta={m['D_delta']:.5f}  RMSE={m['RMSE']:.5f}")
    for p in pairs:
        pre = 10 * np.log10(p.mean_i / p.mean_j)
        post = 10 * np.log10(p.mean_i * g[p.i] / (p.mean_j * g[p.j]))
        print(f"      {names[p.i]} vs {names[p.j]}: n={p.n}  {pre:+.2f} dB -> {post:+.2f} dB")

    print("[4/4] Aplicando ganancias y exportando...")
    outs = []
    for n, path, gi in zip(names, paths, g):
        out = out_dir / f"{n}_RN.tif"
        apply_gain_and_save(path, gi, str(out), cfg)
        outs.append(str(out))
        print(f"      -> {out}")

    report = {
        "gains": dict(zip(names, map(float, g))),
        "gains_dB": dict(zip(names, map(float, 10 * np.log10(g)))),
        "n_pairs_used": len(pairs),
        "isolated_images": isolated,
        "metrics_before": m0, "metrics_after": m1,
        "output_images": outs,
    }
    (out_dir / "rprn_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False))
    print(f"\nReporte: {out_dir / 'rprn_report.json'}")
    return report


if __name__ == "__main__":
    INPUT_DIR = "C:/argenis/RADAR/MOSAICO/2_PROCES/1"        # carpeta con los .dim (o .tif)
    OUTPUT_DIR = "C:/argenis/RADAR/MOSAICO/3_CALIB/1"
    files = sorted(Path(INPUT_DIR).glob("*.dim")) or sorted(Path(INPUT_DIR).glob("*.tif"))
    run([str(f) for f in files], OUTPUT_DIR)

FileNotFoundError: No existe C:\argenis\RADAR\MOSAICO\2_PROCES\1\S1A_IW1_SLC__1SDV_20180823T101751_023372_028AF1_7881.data\Sigma0.img. Bandas disponibles: ['Sigma0_IW1_VV_db']